In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from matplotlib import rcParams
from matplotlib.colors import LogNorm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
rcParams.update({'figure.autolayout': True})
import os
import IPython

In [ ]:
GRAV = 4 * np.pi #AU^3/M_sun/yr^2

#acceleration due to gravity
def g_accel(m,x,y):
    r =  np.sqrt(x**2 + y**2)
    GM = GRAV * m
    ax = -GM * x / r**3
    ay = -GM * y / r**3
    return ax, ay


In [33]:
#parameters to change
theta_e = 0 #earth position in degree angles
dt = 0.1 #year
total_t = 3 #year


#initial conditions
M_sun  = 1 #M_sun
steps = int(total_t / dt)
t = np.zeros(steps + 1)
x = np.zeros(steps + 1)
y = np.zeros(steps + 1)
vx = np.zeros(steps + 1)
vy = np.zeros(steps + 1)

x[0] = 1* np.cos(theta_e)#AU
y[0] = 1* np.sin(theta_e)
vx[0] = 0.0
vy[0] = 2 * np.pi #AU/year

#Eulers Method
for n in range(steps):
    ax,ay = g_accel(M_sun,x[n],y[n])

    vx[n+1] = vx[n] + dt * ax
    vy[n+1] = vy[n] + dt * ay

    x[n+1] = x[n] + dt * vx[n] #using old value of v per Euler
    y[n+1] = y[n] + dt * vy[n]

    t[n+1] = t[n] + dt        

#some other parameters
speed = np.sqrt(vx**2 + vy**2)
r = np.sqrt(x**2 + y**2)
energy = 0.5 * speed**2 - GRAV*M_sun / r

In [48]:
#learning from Matplotlib animation page
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')
ax.set_xlim(1.2 * np.min(x), 1.2 * np.max(x))
ax.set_ylim(1.2 * np.min(y), 1.2 * np.max(y))
ax.set_xlabel("x [AU]",fontsize = 20)
ax.set_ylabel("y [AU]",fontsize = 20)
ax.set_title("Euler's Method",fontsize = 25)

#objects
sun, = ax.plot(0, 0, 'o', markersize=10, label='Sun')
trail, = ax.plot(x[0], y[0], '-', lw=1, label = 'Euler')
earth, = ax.plot(x[0], y[0], 'o', markersize=6)
time = ax.text(0.02, 0.95, '', transform=ax.transAxes,fontsize = 15)
ax.legend()


def update(frame):
    earth.set_data([x[frame]], [y[frame]])
    trail.set_data(x[:frame+1], y[:frame+1])
    time.set_text(f"t = {t[frame]:.2f} yr")
    return earth, trail, time

ani = FuncAnimation(fig,update,frames=len(t),interval=30)

plt.close(fig) 
HTML(ani.to_jshtml())
ani.save("earth_orbit.gif", writer="pillow", fps=20)
